# PROC GAM을 활용한 소매 비선형 수요 곡선 모델링

## 요약

이 노트북은 **PROC GAM**을 사용해 주간 식료품 판매량을 진열가(선반 가격), 매장
기온(계절성 대용 지표), 프로모션 지출의 매끄러운 비선형 함수로 모델링하며,
프로모션 여부에 대한 모수적(파라메트릭) 효과도 함께 반영한다. 일반화가법모형
(GAM)은 선형 회귀로는 놓칠 수 있는 실제 곡선형 가격 탄력성과 계절 수요 형태를
카테고리 매니저가 파악할 수 있게 해 주어, 더 정교한 가격 책정과 프로모션
의사결정을 지원한다.

## 데이터 소스

| 데이터셋 | 행 수 | 단위(Grain) | 주요 변수 | 설명 |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | 주(week) | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | 하나의 대표 식료품 매장에 대한 100주 연속(약 2번의 계절 주기) 합성 주간 판매시점(POS) 데이터. `call streaminit(20250531)`과 `rand()`로 인라인 생성. 주간 판매량은 지수형 가격-반응 곡선, 화씨 72도 부근에서 정점을 이루는 2차 기온/계절성 효과, 오목한(제곱근형) 프로모션 지출 상승 효과, 그리고 이산형 프로모션 주간 플래그로 평균이 결정되는 포아송 수요 과정을 따른다. |

# PROC GAM을 활용한 소매 비선형 수요 곡선 모델링

소매 수요는 가격, 날씨, 프로모션 지출에 대해 직선적으로 반응하는 경우가 드물다.
주력 상품의 소폭 가격 인하는 판매량을 거의 움직이지 못할 수 있는 반면, 심리적
가격대를 넘어서는 순간 판매량이 급격히 뛰어오를 수 있다. 날씨에 따른 수요는
쾌적한 중간 구간에서 정점을 이루다가 양쪽 극단에서는 떨어지며, 프로모션에 따른
판매 상승 효과는 지출이 늘어날수록 체감(수확 체감)한다.

**PROC GAM**(일반화가법모형)은 각 요인을 저마다의 매끄러운 스플라인 항으로
적합시키므로, 경직된 선형 가정이 아니라 데이터 자체가 각 수요 곡선의 형태를
결정하게 한다. 여기서는 하나의 대표 식료품 매장의 100주 연속 주간 판매량을
모델링하며, 프로모션 여부에 대한 모수적 플래그와 가격·기온·프로모션 지출에
대한 평활 스플라인을 포아송 반응 아래에서 결합한다.

## 1단계 — 합성 주간 판매 시계열 생성

하나의 대표 매장에 대해 100주 연속(약 2번의 계절 주기) 판매시점 데이터를
시뮬레이션한다. 데이터 생성 과정은 일부러 비선형으로 설계하여 GAM이 실제와
같은 형태를 복원하는지 확인할 수 있게 한다:

- **가격(Price)**은 지수형 반응 곡선(`exp(-1.1 * Price)`)을 통해 판매량을
  이끌어, 가격이 내려갈수록 수요가 가파르게 상승한다.
- **기온(Temperature)**은 화씨 72도 부근에서 정점을 이루는 2차 곡선으로
  계절성을 대리하며, 쾌적함에 따른 유동인구 변화를 모형화한다.
- **프로모션 지출(PromoSpend)**은 오목한 제곱근형 상승 효과(수확 체감)를
  만든다.
- 이산형 **프로모션(Promotion)** 플래그는 프로모션 주간에 모수적(파라메트릭)
  계단 효과를 더한다.

주간 **판매량(Units)**은 개수 특성을 지닌 판매량의 성격에 맞춰 포아송 분포에서
추출한다.

In [1]:
데이터 store_sales;
   호출 streaminit(20250531);
   길이 Promotion $3;
   반복 Week = 1 까지 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      만약 rand("uniform") < 0.28 이면 반복;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      종료;
      아니면 반복;
         Promotion  = "No";
         PromoSpend = 0;
      종료;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      출력;
   종료;
실행;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## 2단계 — 시뮬레이션 데이터 프로파일링

간단한 `PROC MEANS`로 모델링에 앞서 변수들이 합리적인 소매 범위에 있는지
확인한다: 판매량은 음이 아닌 정수이고, 가격은 몇 달러 안팎에 몰려 있으며,
기온은 계절 전체를 아우르고, 프로모션이 없는 주에는 프로모션 지출이 0이다.

In [2]:
처리 평균 데이터=store_sales n mean MIN MAX maxdec=2;
   변수 Units Price Temperature PromoSpend;
   라벨 Units="판매량" Price="가격" Temperature="기온" PromoSpend="프로모션 지출";
실행;

                                                  The MEANS Procedure

 Variable     Label                       N           Mean     Minimum     Maximum
 ---------------------------------------------------------------------------------
 Units        판매량                       100           7.03        0.00      103.00
 Price        가격                        100           3.15        2.81        3.48
 Temperature  기온                        100          55.50       22.72       83.49
 PromoSpend   프로모션 지출                   100         128.76        0.00      779.00
 ---------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3단계 — 전체 가법 수요 모형 적합

핵심 모형은 다음을 결합한다:

- `param(Promotion)` — `CLASS` 문에서 선언한 프로모션 주간 지시변수에 대한
  모수적(선형) 효과.
- `spline(Price, df=4)` — 곡선형 가격 반응을 포착하는 3차 평활 스플라인.
- `spline(Temperature, df=4)` — 매끄러운 계절성 곡선.
- `spline(PromoSpend, df=3)` — 프로모션 상승 효과의 수확 체감.

판매량이 개수 자료이므로 `dist=poisson`(로그 링크)을 지정한다. `method=gcv`는
일반화 교차검증이 명시적인 자유도 지정 없이도 각 구성 요소의 평활도를 결정하도록
한다. `OUTPUT` 문은 관측치별 예측값과 잔차를 `gam_fit`에 기록한다.

프로시저는 **Null Deviance 2041.12** 대비 **Deviance 43.59**를 보고한다 — 네
가지 평활 및 모수적 요인이 절편만 있는 모형이 남기는 변동의 거의 전부를
설명한다는 뜻이다 — 그리고 **AIC 254.61**을 보고한다. 모수적 `PROMOTIONYES`
추정치는 양수(로그 척도에서 +0.41)로 프로모션에 따른 판매량 상승을 확인해
주며, 가격 스플라인은 강한 음의 선형 추세(−1.71)를 보이는데 이는 우하향하는
수요의 전형적인 특징이다.

In [3]:
처리 gam 데이터=store_sales PLOTS=components(CLM commonaxes);
   분류 Promotion;
   라벨 Units="판매량" Price="가격" Temperature="기온" PromoSpend="프로모션 지출"
         Promotion="프로모션 여부";
   모형 Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   출력 out=gam_fit predicted residual;
실행;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     판매량
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(가격)                       4.0000         4.00


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## 4단계 — 가격·계절성에 집중한 모형

좀 더 실용적인 가격 검토를 위해, 의사결정과 가장 관련이 깊은 두 평활 요인인
가격과 기온만으로 다시 적합하되, 가격에는 심리적 가격대 부근의 굴곡을 풀어낼
수 있도록 추가 유연성(`df=5`)을 부여한다. 프로모션 플래그는 모수적 통제
변수로 그대로 유지한다.

프로모션 지출을 제외하면 **Deviance는 62.80**, **AIC는 269.82**로 올라가며,
둘 다 전체 모형의 43.59, 254.61보다 높다. 모수적 `PROMOTIONYES` 항도 지출
스플라인이 더 이상 신호를 흡수하지 않으므로 프로모션 신호를 더 많이 떠안는다
(+0.41 대비 +1.73). 가격 스플라인은 여전히 음의 추세(−1.51)를 유지하므로,
핵심 수요 이야기는 두 모형 설정 사이에서 안정적이다.

In [4]:
처리 gam 데이터=store_sales PLOTS=components(CLM);
   분류 Promotion;
   라벨 Units="판매량" Price="가격" Temperature="기온" Promotion="프로모션 여부";
   모형 Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
실행;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     판매량
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(가격)                       5.0000         5.0000
Spline(기온)                       4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## 결과 해석

**Regression Model Analysis** 표는 각 구성 요소 내부의 선형 추세와 모수적
프로모션 효과를 보고한다. 양의 `PROMOTIONYES` 계수(전체 모형 +0.41, 축소
모형 +1.73)는 예상된 프로모션 판매량 상승을 확인해 주며, 가격 스플라인의 음의
선형 추세(−1.71, −1.51)는 전형적인 우하향 수요를 반영한다. 기온 스플라인의
작은 양의 선형항(+0.03)은 예상된 결과다: 실제 이야기는 화씨 72도의 쾌적한
정점 부근의 곡률에 있으며, 이는 단일 선형 계수로는 요약할 수 없다.

**Smoothing Model Analysis** 표는 각 스플라인이 소비하는 자유도를 보고한다.
가격과 기온은 각각 유효 자유도 4(축소 모형에서는 가격이 5)를 사용하고
프로모션 지출은 3을 사용한다 — 모두 직선이 사용할 단일 자유도보다 훨씬
높으며, 이것이 바로 선형 회귀가 이런 곡선 관계를 놓치는 이유다.

**Fit Statistics**를 통해 두 모형 설정을 직접 비교할 수 있다. 네 가지 요인을
모두 사용한 전체 모형은 Deviance 43.59, AIC 254.61을 기록하며, 이는 축소
모형의 62.80, 269.82보다 낮다. 두 기준 모두 전체 모형을 선호하며, 이는
프로모션 지출과 프로모션 플래그가 가격과 기온만으로는 설명되지 않는 부분까지
설명력을 더한다는 것을 보여준다. Null Deviance 2041.12에 견주어 보면, 두
모형 모두 수요 변동의 압도적인 대부분을 포착한다.

이 표들을 종합하면 카테고리 매니저에게 정량적이고 데이터 기반의 수요 이야기를
제공한다: 마크다운 폭을 정하는 데 참고할 가파른 가격 반응, 계절적 기온
구간, 그리고 수확 체감형 프로모션 지출 효과 — 단일 선형 탄력성 추정치보다
훨씬 정교한 지침이다. (PROC GAM은 `plots=components` 옵션으로 각 평활 항의
부분예측 곡선도 그려 준다. 위의 수치 구성요소 표가 바로 그 곡선들이 그려지는
근거 자료다.)